# Qwen ADAPT Experiment Notebook

This notebook runs the controlled ADAPT experiment for Qwen models on BIG-Bench Hard and MATH. It compares three conditions under matched sparsity:

1. Unpruned base model
2. Static mask calibrated once and reused for every input
3. ADAPT dynamic masks predicted from runtime activations

Use a Colab Pro GPU runtime. H100 or A100 is preferred; L4 is acceptable for smaller sweeps.

## 1. Setup

In [ ]:
import os
import shutil

REPO_URL = "https://github.com/pketonis/microglia-pruning.git"
REPO_DIR = "/content/microglia-pruning"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed old clone")

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

!pip install -q "torch>=2.3" "transformers>=4.51.0" accelerate bitsandbytes peft datasets scipy numpy tqdm matplotlib seaborn pandas scikit-learn

print("Installation complete")

In [ ]:
import gc
import json
import math
import random
import re
import time
from contextlib import contextmanager, nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import get_dataset_config_names, load_dataset
from tqdm.auto import tqdm

from src.loss import compute_pruning_loss, get_alpha_schedule
from src.pruned_attention import PrunedAttention
from src.system import MicrogliaPruningSystem, _mask_labels_for_padding
from src.utils import set_seed

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
HIDDEN_DIM = 128
TEMPERATURE = 1.0

TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 2
TRAIN_LEARNING_RATE = 1e-6
TRAIN_ALPHA_SCHEDULE = (0.01, 0.2)
TRAIN_MAX_STEPS_PER_EPOCH = 10
TRAIN_MAX_LENGTH = 512
TRAIN_MAX_SAMPLES = 50
VALIDATION_SAMPLES = 5

STATIC_CALIBRATION_SAMPLES = 8
TARGET_KEEP_RATIO = 0.75
MATCHED_SPARSITY_TOLERANCE = 0.02

BBH_MAX_EXAMPLES_PER_TASK = 3
MATH_MAX_EXAMPLES_PER_LEVEL = 3
MAX_NEW_TOKENS_BBH = 32
MAX_NEW_TOKENS_MATH = 64

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = "qwen_adapt_" + RUN_TIMESTAMP
OUTPUT_DIR = Path("/content") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BBH_TASKS = [
    "boolean_expressions",
    "date_understanding",
]

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("SMOKE CONFIG ACTIVE")
print(f"train_epochs={TRAIN_EPOCHS}, train_samples={TRAIN_MAX_SAMPLES}, max_steps={TRAIN_MAX_STEPS_PER_EPOCH}")
print(f"bbh_tasks={BBH_TASKS}, bbh_examples={BBH_MAX_EXAMPLES_PER_TASK}, math_examples_per_level={MATH_MAX_EXAMPLES_PER_LEVEL}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Load Qwen

In [ ]:
system = MicrogliaPruningSystem(
    model=MODEL_ID,
    hidden_dim=HIDDEN_DIM,
    temperature=TEMPERATURE,
    seed=SEED,
)

print(f"Model parameters: {sum(p.numel() for p in system.model.parameters()) / 1e9:.2f}B")
print(f"Agent parameters: {sum(p.numel() for p in system.agents.parameters()) / 1e6:.2f}M")
print(f"Layers: {len(system.get_layers())}")
print(f"Heads: {system.num_heads}")

## 4. Helpers

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_attention_modules(system):
    modules = []
    for layer in system.get_layers():
        attn = getattr(layer, "self_attn", getattr(layer, "attn", None))
        if isinstance(attn, PrunedAttention):
            modules.append(attn)
    return modules


def get_last_mask_matrix(system):
    masks = []
    for attn in get_attention_modules(system):
        if attn.last_masks is None:
            return None
        masks.append(attn.last_masks.detach().float().mean(dim=0).cpu().numpy())
    if not masks:
        return None
    return np.stack(masks)


def mean_keep_ratio(mask_matrices):
    if len(mask_matrices) == 0:
        return float("nan")
    values = np.asarray([np.asarray(mask, dtype=np.float32).mean() for mask in mask_matrices])
    if not np.isfinite(values).all():
        raise RuntimeError("Non-finite keep ratio detected")
    return float(values.mean())


def project_mask_keep_ratio(mask_matrix, keep_ratio):
    mask_matrix = np.asarray(mask_matrix, dtype=np.float32)
    projected = np.zeros_like(mask_matrix, dtype=np.float32)
    keep_count = max(1, int(round(float(keep_ratio) * mask_matrix.shape[1])))
    for layer_idx in range(mask_matrix.shape[0]):
        indices = np.argsort(mask_matrix[layer_idx])[-keep_count:]
        projected[layer_idx, indices] = 1.0
    return projected


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()


def extract_final_answer(text):
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].strip()
    marker = re.findall(r"####\s*([^\n]+)", text)
    if marker:
        return marker[-1].strip()
    options = re.findall(r"\(([A-Z])\)", text)
    if options:
        return f"({options[-1]})"
    booleans = re.findall(r"\b(true|false|yes|no)\b", text, flags=re.IGNORECASE)
    if booleans:
        value = booleans[-1].lower()
        return "True" if value == "true" else "False" if value == "false" else value.capitalize()
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    if numbers:
        return numbers[-1]
    return text.strip()


def score_exact_or_numeric(prediction, gold):
    pred_answer = extract_final_answer(prediction)
    gold_answer = extract_final_answer(gold)
    pred_numbers = re.findall(r"-?\d+(?:\.\d+)?", str(pred_answer).replace(",", ""))
    gold_numbers = re.findall(r"-?\d+(?:\.\d+)?", str(gold_answer).replace(",", ""))
    if pred_numbers and gold_numbers:
        return int(abs(float(pred_numbers[-1]) - float(gold_numbers[-1])) < 1e-6)
    return int(normalize_text(pred_answer) == normalize_text(gold_answer))


def format_eval_prompt(system, row):
    question = str(row["question"])
    if "task" in row:
        content = "Answer the following BIG-Bench Hard question. Provide only the final answer.\n\n" + question
        if system._has_chat_template:
            return system.tokenizer.apply_chat_template(
                [{"role": "user", "content": content}],
                tokenize=False,
                add_generation_prompt=True,
            )
        return content + "\nAnswer:"
    return question


def save_json(path, payload):
    with Path(path).open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2)


def select_first(dataset, count):
    return dataset.select(range(min(count, len(dataset))))

## 5. Train ADAPT Agents

This training loop uses forward hooks on `MicrogliaAgent` to keep live mask tensors in the sparsity loss path. That avoids using detached `last_masks` for regularization.

In [ ]:
def train_adapt_agents(
    system,
    dataset_name="gsm8k",
    num_epochs=3,
    batch_size=2,
    learning_rate=1e-4,
    alpha_schedule=(0.01, 0.2),
    max_steps_per_epoch=300,
    max_samples=1000,
    max_length=512,
    validation_samples=50,
    precision="bf16",
):
    system.model.requires_grad_(False)
    if not system.wrapped:
        system._wrap_attention_layers()
    system.agents.requires_grad_(True)
    system._enable_pruning(True)
    system._set_budget_keep_ratio(None)

    dataset = load_dataset(dataset_name, "main")
    train_split = select_first(dataset["train"], max_samples)
    val_split = select_first(dataset["test"], validation_samples)

    processed = []
    for row in tqdm(train_split, desc="Tokenizing train"):
        text = system._format_train_sample(row["question"], row["answer"])
        encoded = system.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        processed.append({
            "input_ids": encoded["input_ids"][0],
            "attention_mask": encoded["attention_mask"][0],
        })

    loader = torch.utils.data.DataLoader(processed, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(system.agents.parameters(), lr=learning_rate, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    pad_token_id = getattr(system.tokenizer, "pad_token_id", None)

    if precision not in {"fp32", "fp16", "bf16"}:
        raise ValueError("precision must be fp32, fp16, or bf16")
    amp_dtype = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[precision]
    scaler = torch.amp.GradScaler("cuda", enabled=(precision == "fp16" and torch.cuda.is_available()))

    live_masks = []
    handles = []

    def capture_agent_output(module, inputs, output):
        live_masks.append(output)

    for agent in system.agents:
        handles.append(agent.register_forward_hook(capture_agent_output))

    best_state = None
    best_val_loss = float("inf")
    history = []

    try:
        for epoch in range(num_epochs):
            alpha = get_alpha_schedule(epoch, num_epochs, alpha_schedule[0], alpha_schedule[1])
            system.model.train()
            totals = {"task_loss": 0.0, "sparsity_loss": 0.0, "total_loss": 0.0}
            steps = 0

            progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
            for step, batch in enumerate(progress):
                batch = {key: value.to(system.device) for key, value in batch.items()}
                labels = _mask_labels_for_padding(
                    batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    pad_token_id=pad_token_id,
                )
                live_masks.clear()
                optimizer.zero_grad(set_to_none=True)
                system.model.zero_grad(set_to_none=True)

                if precision in {"fp16", "bf16"} and torch.cuda.is_available():
                    amp_context = torch.autocast(device_type="cuda", dtype=amp_dtype)
                elif precision == "bf16":
                    amp_context = torch.autocast(device_type="cpu", dtype=torch.bfloat16)
                else:
                    amp_context = nullcontext()

                with amp_context:
                    outputs = system.model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                    )

                logits = outputs.logits.float()
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels[:, 1:].contiguous()
                task_loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1),
                    ignore_index=-100,
                )
                if not torch.isfinite(task_loss):
                    print("Skipping batch with non-finite task loss")
                    continue
                if not live_masks:
                    continue
                masks = torch.cat(live_masks, dim=0)
                if not torch.isfinite(masks).all():
                    raise RuntimeError("Agent produced non-finite masks during training")
                # Smoke training keeps task loss as a diagnostic and trains agents
                # only through mask regularization. Backpropagating task loss
                # through all pruned Qwen layers is numerically unstable here.
                loss_dict = compute_pruning_loss(task_loss.detach(), masks, alpha=alpha)
                total_loss = loss_dict["total_loss"]
                if not torch.isfinite(total_loss):
                    raise RuntimeError("Training loss became non-finite")

                totals["task_loss"] += loss_dict["task_loss"]
                totals["sparsity_loss"] += loss_dict["sparsity_loss"]
                totals["total_loss"] += float(total_loss.detach().cpu().item())
                steps += 1
                mask_loss_value = loss_dict["sparsity_loss"]
                progress.set_postfix({"loss": f"{total_loss.item():.3f}", "mask": f"{mask_loss_value:.3f}"})

                if scaler.is_enabled():
                    scaler.scale(total_loss).backward()
                    scaler.unscale_(optimizer)
                    grad_norm = torch.nn.utils.clip_grad_norm_(system.agents.parameters(), 0.1)
                    if torch.isfinite(grad_norm):
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        scaler.update()
                        optimizer.zero_grad(set_to_none=True)
                        print("Skipping non-finite gradient update")
                        continue
                else:
                    total_loss.backward()
                    grad_norm = torch.nn.utils.clip_grad_norm_(system.agents.parameters(), 0.1)
                    if torch.isfinite(grad_norm):
                        optimizer.step()
                    else:
                        optimizer.zero_grad(set_to_none=True)
                        print("Skipping non-finite gradient update")
                        continue

                if max_steps_per_epoch is not None and step + 1 >= max_steps_per_epoch:
                    break

            if steps == 0:
                raise RuntimeError("No finite training steps completed")
            scheduler.step()
            averages = {key: value / steps for key, value in totals.items()}

            system.model.eval()
            val_losses = []
            with torch.no_grad():
                for row in tqdm(val_split, desc="Validation"):
                    text = system._format_train_sample(row["question"], row["answer"])
                    encoded = system.tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt").to(system.device)
                    labels = _mask_labels_for_padding(
                        encoded["input_ids"],
                        attention_mask=encoded["attention_mask"],
                        pad_token_id=pad_token_id,
                    )
                    outputs = system.model(input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"])
                    logits = outputs.logits.float()
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
                    val_loss = F.cross_entropy(
                        shift_logits.view(-1, shift_logits.size(-1)),
                        shift_labels.view(-1),
                        ignore_index=-100,
                    )
                    if torch.isfinite(val_loss):
                        val_losses.append(float(val_loss.detach().cpu().item()))

            averages["val_loss"] = float(np.mean(val_losses)) if val_losses else float("inf")
            history.append(averages)
            print(f"Epoch {epoch + 1}: {averages}")

            if averages["val_loss"] < best_val_loss:
                best_val_loss = averages["val_loss"]
                best_state = {key: value.detach().cpu().clone() for key, value in system.agents.state_dict().items()}

            clear_memory()

        if best_state is not None:
            system.agents.load_state_dict(best_state)
        system.training_history = history
        return history
    finally:
        for handle in handles:
            handle.remove()
        system._enable_pruning(False)


history = train_adapt_agents(
    system,
    num_epochs=TRAIN_EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    learning_rate=TRAIN_LEARNING_RATE,
    alpha_schedule=TRAIN_ALPHA_SCHEDULE,
    max_steps_per_epoch=TRAIN_MAX_STEPS_PER_EPOCH,
    max_samples=TRAIN_MAX_SAMPLES,
    max_length=TRAIN_MAX_LENGTH,
    validation_samples=VALIDATION_SAMPLES,
    precision="bf16",
)

save_json(OUTPUT_DIR / "training_history.json", history)
system.save_checkpoint(str(OUTPUT_DIR / "adapt_agents.pt"))

## 6. Static Mask Calibration

In [ ]:
class ConstantMaskAgent(torch.nn.Module):
    def __init__(self, mask):
        super().__init__()
        mask = np.asarray(mask, dtype=np.float32)
        if not np.isfinite(mask).all():
            raise ValueError("Static mask contains non-finite values")
        self.register_buffer("mask", torch.tensor(mask, dtype=torch.float32))
        self.num_heads = int(self.mask.numel())
        self.temperature = 1.0

    def forward(self, activation_stats, layer_idx=None):
        return self.mask.to(device=activation_stats.device, dtype=torch.float32).unsqueeze(0).expand(activation_stats.shape[0], -1)


@contextmanager
def use_static_masks(system, static_masks):
    modules = get_attention_modules(system)
    original_agents = [module.agent for module in modules]
    try:
        for module, mask in zip(modules, static_masks):
            module.agent = ConstantMaskAgent(mask).to(system.device)
        yield
    finally:
        for module, agent in zip(modules, original_agents):
            module.agent = agent


def collect_calibration_prompts(count):
    dataset = load_dataset("gsm8k", "main", split="train")
    dataset = select_first(dataset, count)
    return [row["question"] for row in dataset]


def calibrate_static_masks(system, prompts, max_new_tokens=1):
    if not system.wrapped:
        system._wrap_attention_layers()
    system.model.eval()
    matrices = []
    for prompt in tqdm(prompts, desc="Calibrating static masks"):
        _ = system.generate(prompt, max_new_tokens=max_new_tokens, use_pruning=True)
        matrix = get_last_mask_matrix(system)
        if matrix is not None:
            matrices.append(matrix)
    if not matrices:
        raise RuntimeError("No calibration masks were captured")
    stacked = np.stack(matrices)
    if not np.isfinite(stacked).all():
        raise RuntimeError("Calibration produced non-finite masks")
    return np.mean(stacked, axis=0), matrices


calibration_prompts = collect_calibration_prompts(STATIC_CALIBRATION_SAMPLES)
raw_static_masks, calibration_mask_log = calibrate_static_masks(system, calibration_prompts)
calibration_keep_ratio = float(raw_static_masks.mean())

if TARGET_KEEP_RATIO is None:
    TARGET_KEEP_RATIO = calibration_keep_ratio
static_masks = project_mask_keep_ratio(raw_static_masks, TARGET_KEEP_RATIO)

np.save(OUTPUT_DIR / "static_masks.npy", static_masks)
np.save(OUTPUT_DIR / "calibration_mask_log.npy", np.stack(calibration_mask_log))

print(f"Static calibration keep ratio: {calibration_keep_ratio:.3f}")
print(f"Target keep ratio: {TARGET_KEEP_RATIO:.3f}")

## 7. Dataset Loaders

In [ ]:
def load_bbh_task(task_name, max_examples):
    try:
        dataset = load_dataset("lukaemon/bbh", task_name, split="test")
    except Exception:
        loaded = load_dataset("lukaemon/bbh")
        split = loaded["test"] if "test" in loaded else loaded[list(loaded.keys())[0]]
        if "task" in split.column_names:
            split = split.filter(lambda row: row.get("task") == task_name)
        dataset = split
    dataset = select_first(dataset, max_examples)
    rows = []
    for row in dataset:
        question = row.get("input", row.get("question", row.get("prompt", "")))
        answer = row.get("target", row.get("answer", row.get("label", "")))
        rows.append({"task": task_name, "question": question, "answer": answer})
    return rows


def parse_math_level(level):
    match = re.search(r"\d+", str(level))
    return int(match.group(0)) if match else None


def load_math_by_level(max_examples_per_level):
    grouped = {level: [] for level in range(1, 6)}
    configs = get_dataset_config_names("EleutherAI/hendrycks_math")
    for config in configs:
        dataset = load_dataset("EleutherAI/hendrycks_math", config, split="test")
        for row in dataset:
            level = parse_math_level(row.get("level", ""))
            if level in grouped and len(grouped[level]) < max_examples_per_level:
                grouped[level].append({
                    "level": level,
                    "question": row.get("problem", ""),
                    "answer": row.get("solution", ""),
                })
        if all(len(rows) >= max_examples_per_level for rows in grouped.values()):
            break
    return grouped

## 8. Evaluation Core

In [ ]:
def generate_condition(system, prompt, condition, static_masks=None, max_new_tokens=64, apply_chat_template=True):
    if condition == "unpruned":
        return system.generate(prompt, max_new_tokens=max_new_tokens, use_pruning=False, apply_chat_template=apply_chat_template)
    if condition == "adapt":
        return system.generate(
            prompt,
            max_new_tokens=max_new_tokens,
            use_pruning=True,
            budget_keep_ratio=TARGET_KEEP_RATIO,
            apply_chat_template=apply_chat_template,
        )
    if condition == "static":
        with use_static_masks(system, static_masks):
            return system.generate(
                prompt,
                max_new_tokens=max_new_tokens,
                use_pruning=True,
                budget_keep_ratio=TARGET_KEEP_RATIO,
                apply_chat_template=apply_chat_template,
            )
    raise ValueError(f"Unknown condition: {condition}")


def evaluate_rows(system, rows, condition, static_masks=None, max_new_tokens=64, label="eval"):
    outcomes = []
    mask_matrices = []
    records = []
    for index, row in enumerate(tqdm(rows, desc=label)):
        prompt = format_eval_prompt(system, row)
        apply_chat_template = "task" not in row
        start = time.perf_counter()
        prediction = generate_condition(
            system,
            prompt,
            condition,
            static_masks=static_masks,
            max_new_tokens=max_new_tokens,
            apply_chat_template=apply_chat_template,
        )
        latency_ms = (time.perf_counter() - start) * 1000.0
        correct = score_exact_or_numeric(prediction, row["answer"])
        outcomes.append(correct)

        if condition in {"adapt", "static"}:
            matrix = get_last_mask_matrix(system)
            if matrix is not None:
                mask_matrices.append(matrix)

        records.append({
            "index": index,
            "condition": condition,
            "correct": correct,
            "latency_ms": latency_ms,
            "question": row["question"],
            "answer": row["answer"],
            "prediction": prediction,
        })

    result = {
        "condition": condition,
        "accuracy": float(np.mean(outcomes)) if outcomes else 0.0,
        "correct": int(np.sum(outcomes)),
        "total": int(len(outcomes)),
        "mean_latency_ms": float(np.mean([record["latency_ms"] for record in records])) if records else float("nan"),
        "mean_keep_ratio": mean_keep_ratio(mask_matrices) if mask_matrices else 1.0,
    }
    return result, records, mask_matrices


def assert_matched_sparsity(static_keep_ratio, adapt_keep_ratio, tolerance=MATCHED_SPARSITY_TOLERANCE):
    if not np.isfinite([static_keep_ratio, adapt_keep_ratio]).all():
        raise AssertionError("Static and ADAPT keep ratios must be finite")
    delta = abs(static_keep_ratio - adapt_keep_ratio)
    if delta > tolerance:
        raise AssertionError(
            f"Static and ADAPT keep ratios differ by {delta:.3f}; "
            f"static={static_keep_ratio:.3f}, adapt={adapt_keep_ratio:.3f}"
        )

## 9. BIG-Bench Hard Sweep

In [ ]:
bbh_results = []
bbh_records = []
bbh_mask_logs = {}

for task_name in BBH_TASKS:
    print(f"\nTask: {task_name}")
    try:
        rows = load_bbh_task(task_name, BBH_MAX_EXAMPLES_PER_TASK)
    except Exception as exc:
        print(f"Skipping {task_name}: {exc}")
        continue
    if not rows:
        print(f"Skipping {task_name}: no rows")
        continue

    task_results = []
    for condition in ["unpruned", "static", "adapt"]:
        result, records, masks = evaluate_rows(
            system,
            rows,
            condition,
            static_masks=static_masks,
            max_new_tokens=MAX_NEW_TOKENS_BBH,
            label=f"{task_name}:{condition}",
        )
        result["task"] = task_name
        task_results.append(result)
        bbh_records.extend([{**record, "task": task_name} for record in records])
        if masks:
            bbh_mask_logs[f"{task_name}_{condition}"] = np.stack(masks)
        clear_memory()

    by_condition = {row["condition"]: row for row in task_results}
    if "static" in by_condition and "adapt" in by_condition:
        try:
            assert_matched_sparsity(by_condition["static"]["mean_keep_ratio"], by_condition["adapt"]["mean_keep_ratio"])
            matched = True
        except AssertionError as exc:
            print(f"Matched sparsity warning: {exc}")
            matched = False
        for row in task_results:
            row["matched_sparsity"] = matched

    bbh_results.extend(task_results)
    pd.DataFrame(bbh_results).to_csv(OUTPUT_DIR / "bbh_results_partial.csv", index=False)

bbh_results_df = pd.DataFrame(bbh_results)
bbh_records_df = pd.DataFrame(bbh_records)
bbh_results_df.to_csv(OUTPUT_DIR / "bbh_results.csv", index=False)
bbh_records_df.to_csv(OUTPUT_DIR / "bbh_predictions.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "bbh_mask_logs.npz", **bbh_mask_logs)

bbh_results_df

## 10. MATH Difficulty Sweep

In [ ]:
math_by_level = load_math_by_level(MATH_MAX_EXAMPLES_PER_LEVEL)
math_results = []
math_records = []
math_mask_logs = {}

for level, rows in math_by_level.items():
    print(f"\nMATH Level {level}")
    if not rows:
        continue
    level_results = []
    for condition in ["unpruned", "static", "adapt"]:
        result, records, masks = evaluate_rows(
            system,
            rows,
            condition,
            static_masks=static_masks,
            max_new_tokens=MAX_NEW_TOKENS_MATH,
            label=f"math_level_{level}:{condition}",
        )
        result["level"] = level
        level_results.append(result)
        math_records.extend([{**record, "level": level} for record in records])
        if masks:
            math_mask_logs[f"level_{level}_{condition}"] = np.stack(masks)
        clear_memory()

    by_condition = {row["condition"]: row for row in level_results}
    if "static" in by_condition and "adapt" in by_condition:
        try:
            assert_matched_sparsity(by_condition["static"]["mean_keep_ratio"], by_condition["adapt"]["mean_keep_ratio"])
            matched = True
        except AssertionError as exc:
            print(f"Matched sparsity warning: {exc}")
            matched = False
        for row in level_results:
            row["matched_sparsity"] = matched

    math_results.extend(level_results)
    pd.DataFrame(math_results).to_csv(OUTPUT_DIR / "math_results_partial.csv", index=False)

math_results_df = pd.DataFrame(math_results)
math_records_df = pd.DataFrame(math_records)
math_results_df.to_csv(OUTPUT_DIR / "math_results.csv", index=False)
math_records_df.to_csv(OUTPUT_DIR / "math_predictions.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "math_mask_logs.npz", **math_mask_logs)

math_results_df

## 11. Figures

In [ ]:
sns.set_theme(style="whitegrid")

if not bbh_results_df.empty:
    pivot = bbh_results_df.pivot(index="task", columns="condition", values="accuracy").reset_index()
    if "adapt" in pivot and "static" in pivot:
        pivot["adapt_minus_static"] = pivot["adapt"] - pivot["static"]
        pivot = pivot.sort_values("adapt_minus_static", ascending=False)
        plt.figure(figsize=(12, 7))
        sns.barplot(data=pivot, x="adapt_minus_static", y="task", color="#4477AA")
        plt.axvline(0, color="black", linewidth=1)
        plt.xlabel("ADAPT - Static Accuracy")
        plt.ylabel("BBH Task")
        plt.title("Task-Level Accuracy Gap")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "figure_1_bbh_adapt_minus_static.png", dpi=300)
        plt.show()

if not math_results_df.empty:
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=math_results_df, x="level", y="accuracy", hue="condition", marker="o")
    plt.xlabel("MATH Difficulty Level")
    plt.ylabel("Accuracy")
    plt.title("MATH Accuracy by Difficulty")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_2_math_accuracy_by_level.png", dpi=300)
    plt.show()

def flatten_keep_ratios(mask_log):
    rows = []
    for name, values in mask_log.items():
        arr = np.asarray(values)
        for idx in range(arr.shape[0]):
            rows.append({"group": name, "keep_ratio": float(arr[idx].mean())})
    return pd.DataFrame(rows)

math_keep_df = flatten_keep_ratios({key: value for key, value in math_mask_logs.items() if key.endswith("adapt")})
if not math_keep_df.empty:
    math_keep_df["difficulty"] = math_keep_df["group"].str.extract(r"level_(\d+)_adapt").astype(int)
    math_keep_df["band"] = np.where(math_keep_df["difficulty"] <= 2, "easy", np.where(math_keep_df["difficulty"] >= 4, "hard", "middle"))
    plot_df = math_keep_df[math_keep_df["band"].isin(["easy", "hard"])]
    plt.figure(figsize=(8, 5))
    sns.histplot(data=plot_df, x="keep_ratio", hue="band", bins=20, stat="density", common_norm=False, element="step")
    plt.xlabel("Per-Input Keep Ratio")
    plt.ylabel("Density")
    plt.title("ADAPT Keep Ratio Distribution")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_3_keep_ratio_distribution.png", dpi=300)
    plt.show()

bbh_task_masks = []
bbh_task_names = []
for key, values in bbh_mask_logs.items():
    if key.endswith("adapt"):
        bbh_task_names.append(key.replace("_adapt", ""))
        bbh_task_masks.append(np.asarray(values).mean(axis=0).reshape(-1))

if len(bbh_task_masks) >= 2:
    matrix = np.stack(bbh_task_masks)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True).clip(min=1e-8)
    sim = (matrix / norms) @ (matrix / norms).T
    plt.figure(figsize=(10, 8))
    sns.heatmap(sim, xticklabels=bbh_task_names, yticklabels=bbh_task_names, cmap="viridis", vmin=0, vmax=1)
    plt.title("ADAPT Mask Similarity Across BBH Tasks")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_4_bbh_mask_similarity.png", dpi=300)
    plt.show()

## 12. Export

In [ ]:
summary = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "seed": SEED,
    "output_dir": str(OUTPUT_DIR),
    "target_keep_ratio": float(TARGET_KEEP_RATIO),
    "static_calibration_keep_ratio": float(calibration_keep_ratio),
    "bbh_tasks_run": sorted(bbh_results_df["task"].unique().tolist()) if not bbh_results_df.empty else [],
    "math_levels_run": sorted(math_results_df["level"].unique().tolist()) if not math_results_df.empty else [],
}

save_json(OUTPUT_DIR / "run_summary.json", summary)

!cd /content && zip -qr {RUN_NAME}.zip {RUN_NAME}
print(f"Saved outputs to {OUTPUT_DIR}")
print(f"Archive: /content/{RUN_NAME}.zip")